# DGL vs PyG — TxGNN Equivalence Check

This notebook verifies that the PyG refactoring of TxGNN produces **logically identical** results to the original DGL implementation.

Three verification stages:
1. **Graph construction parity** — same number of nodes, edges, and edge index values
2. **Single forward-pass parity** — given identical weights, both models produce the same scores
3. **Training metric parity** — after N epochs with the same seed, AUROC/AUPRC agree within tolerance


In [ ]:
import sys, os
import torch
import numpy as np
import pandas as pd

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd()))

# Add both implementations to the path
DGL_ROOT = os.path.join(REPO_ROOT, 'dgl_implementation')
PYG_ROOT = os.path.join(REPO_ROOT, 'pyg_implementation')

print('Repo root :', REPO_ROOT)
print('DGL impl  :', DGL_ROOT)
print('PyG impl  :', PYG_ROOT)


## 0  — Shared config

In [ ]:
DATA_FOLDER = os.path.join(REPO_ROOT, 'data')   # shared data folder
SPLIT       = 'complex_disease'
SEED        = 42
DEVICE      = 'cuda:0' if torch.cuda.is_available() else 'cpu'
N_HID       = 100
N_INP       = 100
N_OUT       = 100
PROTO       = True
PROTO_NUM   = 5
ATTENTION   = False

print('Device:', DEVICE)


---
## Stage 1 — Graph construction parity

Load data with both implementations and compare the resulting graphs.

In [ ]:
# ── DGL graph ───────────────────────────────────────────────
if DGL_ROOT not in sys.path:
    sys.path.insert(0, DGL_ROOT)

import importlib
# isolate the two implementations by importing under aliases
import txgnn as txgnn_dgl
importlib.reload(txgnn_dgl)

data_dgl = txgnn_dgl.TxData(data_folder_path=DATA_FOLDER)
data_dgl.prepare_split(split=SPLIT, seed=SEED)
G_dgl = data_dgl.G
print('DGL graph loaded')
print('  ntypes :', G_dgl.ntypes)
print('  etypes :', len(G_dgl.canonical_etypes), 'canonical edge types')


In [ ]:
# ── PyG graph ───────────────────────────────────────────────
# Remove DGL impl from path, add PyG impl
if DGL_ROOT in sys.path:
    sys.path.remove(DGL_ROOT)
if PYG_ROOT not in sys.path:
    sys.path.insert(0, PYG_ROOT)

# Need to reload since 'txgnn' module name is the same
for mod_name in list(sys.modules.keys()):
    if mod_name.startswith('txgnn'):
        del sys.modules[mod_name]

import txgnn as txgnn_pyg

data_pyg = txgnn_pyg.TxData(data_folder_path=DATA_FOLDER)
data_pyg.prepare_split(split=SPLIT, seed=SEED)
G_pyg = data_pyg.G
print('PyG graph loaded')
print('  node_types :', G_pyg.node_types)
print('  edge_types :', len(G_pyg.edge_types), 'canonical edge types')


In [ ]:
# ── Compare node counts ────────────────────────────────────
print('=== Node count parity ===')
all_pass = True
for ntype in G_dgl.ntypes:
    n_dgl = G_dgl.number_of_nodes(ntype)
    n_pyg = G_pyg[ntype].num_nodes
    status = '✓' if n_dgl == n_pyg else '✗'
    if n_dgl != n_pyg:
        all_pass = False
    print(f'  {status}  {ntype:30s}  DGL={n_dgl:6d}  PyG={n_pyg:6d}')
print('All pass:', all_pass)


In [ ]:
# ── Compare edge counts ────────────────────────────────────
print('=== Edge count parity ===')
all_pass = True
for et in G_dgl.canonical_etypes:
    e_dgl = G_dgl.number_of_edges(et)
    e_pyg = G_pyg[et].num_edges
    status = '✓' if e_dgl == e_pyg else '✗'
    if e_dgl != e_pyg:
        all_pass = False
    print(f'  {status}  {str(et):55s}  DGL={e_dgl:7d}  PyG={e_pyg:7d}')
print('All pass:', all_pass)


In [ ]:
# ── Compare edge indices ────────────────────────────────────
print('=== Edge index parity (sorted) ===')
all_pass = True
for et in G_dgl.canonical_etypes:
    src_dgl, dst_dgl = G_dgl.edges(etype=et)
    src_dgl = src_dgl.numpy()
    dst_dgl = dst_dgl.numpy()
    
    ei_pyg = G_pyg[et].edge_index
    src_pyg = ei_pyg[0].numpy()
    dst_pyg = ei_pyg[1].numpy()
    
    # Sort both by (src, dst) for comparison
    idx_dgl = np.lexsort((dst_dgl, src_dgl))
    idx_pyg = np.lexsort((dst_pyg, src_pyg))
    
    match = (np.array_equal(src_dgl[idx_dgl], src_pyg[idx_pyg]) and
             np.array_equal(dst_dgl[idx_dgl], dst_pyg[idx_pyg]))
    status = '✓' if match else '✗'
    if not match:
        all_pass = False
    print(f'  {status}  {str(et):55s}')
print('All pass:', all_pass)


---
## Stage 2 — Single forward-pass parity

Initialize both models with the **same random seed** and **copy weights** from DGL to PyG. Then check that scores match.

In [ ]:
torch.manual_seed(0)
np.random.seed(0)

# ── DGL model ─────────────────────────────────────────────
if PYG_ROOT in sys.path:
    sys.path.remove(PYG_ROOT)
if DGL_ROOT not in sys.path:
    sys.path.insert(0, DGL_ROOT)
for mod in list(sys.modules.keys()):
    if mod.startswith('txgnn'):
        del sys.modules[mod]
import txgnn as txgnn_dgl

model_dgl = txgnn_dgl.TxGNN(data=data_dgl, device=DEVICE)
model_dgl.model_initialize(
    n_hid=N_HID, n_inp=N_INP, n_out=N_OUT,
    proto=PROTO, proto_num=PROTO_NUM,
    attention=ATTENTION,
    sim_measure='all_nodes_profile',
    agg_measure='rarity',
)
print('DGL model initialized, params:', sum(p.numel() for p in model_dgl.model.parameters()))


In [ ]:
torch.manual_seed(0)
np.random.seed(0)

# ── PyG model ─────────────────────────────────────────────
if DGL_ROOT in sys.path:
    sys.path.remove(DGL_ROOT)
if PYG_ROOT not in sys.path:
    sys.path.insert(0, PYG_ROOT)
for mod in list(sys.modules.keys()):
    if mod.startswith('txgnn'):
        del sys.modules[mod]
import txgnn as txgnn_pyg

model_pyg = txgnn_pyg.TxGNN(data=data_pyg, device=DEVICE)
model_pyg.model_initialize(
    n_hid=N_HID, n_inp=N_INP, n_out=N_OUT,
    proto=PROTO, proto_num=PROTO_NUM,
    attention=ATTENTION,
    sim_measure='all_nodes_profile',
    agg_measure='rarity',
)
print('PyG model initialized, params:', sum(p.numel() for p in model_pyg.model.parameters()))


In [ ]:
# ── Copy DGL weights into PyG model ───────────────────────
# Parameters must match in name and shape for a valid comparison.
# We'll compare the parameter count and then do a name-by-name check.

dgl_params = dict(model_dgl.model.named_parameters())
pyg_params = dict(model_pyg.model.named_parameters())

print(f'DGL param tensors: {len(dgl_params)}')
print(f'PyG param tensors: {len(pyg_params)}')

# Show any name mismatches
dgl_names = set(dgl_params.keys())
pyg_names = set(pyg_params.keys())
only_dgl = dgl_names - pyg_names
only_pyg = pyg_names - dgl_names
common   = dgl_names & pyg_names

print(f'\nCommon params : {len(common)}')
print(f'Only in DGL   : {sorted(only_dgl)}')
print(f'Only in PyG   : {sorted(only_pyg)}')


In [ ]:
# Copy common weights + check shape compatibility
shape_mismatches = []
with torch.no_grad():
    for name in sorted(common):
        if dgl_params[name].shape == pyg_params[name].shape:
            pyg_params[name].copy_(dgl_params[name])
        else:
            shape_mismatches.append((name, dgl_params[name].shape, pyg_params[name].shape))

if shape_mismatches:
    print('Shape mismatches (cannot copy):')
    for n, s_d, s_p in shape_mismatches:
        print(f'  {n}: DGL {s_d}  PyG {s_p}')
else:
    print('All common weights copied successfully — shapes match.')


In [ ]:
# ── Also copy node embeddings (inp) ───────────────────────
# DGL stores inp on G.nodes[ntype].data['inp']
# PyG stores inp on G[ntype].inp
with torch.no_grad():
    for ntype in data_dgl.G.ntypes:
        dgl_emb = data_dgl.G.nodes[ntype].data['inp']
        data_pyg.G[ntype].inp = dgl_emb.clone()
        model_pyg.G[ntype].inp = dgl_emb.clone()
print('Node embeddings synced from DGL → PyG')


In [ ]:
# ── Forward pass comparison ────────────────────────────────
from txgnn.utils import Full_Graph_NegSampler  # loaded from whichever is on path

# DGL forward
if PYG_ROOT in sys.path: sys.path.remove(PYG_ROOT)
if DGL_ROOT not in sys.path: sys.path.insert(0, DGL_ROOT)
for mod in list(sys.modules.keys()):
    if mod.startswith('txgnn'): del sys.modules[mod]
import txgnn as txgnn_dgl
from txgnn.utils import Full_Graph_NegSampler as FNS_dgl

torch.manual_seed(1)
G_dgl_dev = data_dgl.G.to(DEVICE)
neg_sampler_dgl = FNS_dgl(G_dgl_dev, 1, 'fix_dst', DEVICE)
neg_G_dgl = neg_sampler_dgl(G_dgl_dev)

model_dgl.model.eval()
with torch.no_grad():
    scores_dgl, scores_neg_dgl, pos_dgl, neg_dgl = model_dgl.model(
        G_dgl_dev, neg_G_dgl, pretrain_mode=False, mode='test'
    )
print('DGL forward done')


In [ ]:
# PyG forward
if DGL_ROOT in sys.path: sys.path.remove(DGL_ROOT)
if PYG_ROOT not in sys.path: sys.path.insert(0, PYG_ROOT)
for mod in list(sys.modules.keys()):
    if mod.startswith('txgnn'): del sys.modules[mod]
import txgnn as txgnn_pyg
from txgnn.utils import Full_Graph_NegSampler as FNS_pyg

torch.manual_seed(1)   # same seed → same negative edges
G_pyg_dev = data_pyg.G.to(DEVICE)
neg_sampler_pyg = FNS_pyg(G_pyg_dev, 1, 'fix_dst', DEVICE)
neg_G_pyg = neg_sampler_pyg(G_pyg_dev)

model_pyg.model.eval()
with torch.no_grad():
    scores_pyg, scores_neg_pyg, pos_pyg, neg_pyg = model_pyg.model(
        G_pyg_dev, neg_G_pyg, pretrain_mode=False, mode='test'
    )
print('PyG forward done')


In [ ]:
# ── Compare positive scores per edge type ─────────────────
print('=== Positive score parity ===')
dd_etypes = [
    ('drug', 'contraindication', 'disease'),
    ('drug', 'indication', 'disease'),
    ('drug', 'off-label use', 'disease'),
    ('disease', 'rev_contraindication', 'drug'),
    ('disease', 'rev_indication', 'drug'),
    ('disease', 'rev_off-label use', 'drug'),
]

TOL = 1e-5
all_pass = True
for et in dd_etypes:
    if et not in scores_dgl or et not in scores_pyg:
        print(f'  SKIP  {et}  (not in one of the score dicts)')
        continue
    s_d = scores_dgl[et].detach().cpu()
    s_p = scores_pyg[et].detach().cpu()
    if s_d.shape != s_p.shape:
        print(f'  ✗  {et}  shape mismatch DGL={s_d.shape} PyG={s_p.shape}')
        all_pass = False
        continue
    max_diff = (s_d - s_p).abs().max().item()
    ok = max_diff < TOL
    if not ok: all_pass = False
    print(f'  {"✓" if ok else "✗"}  {str(et):55s}  max_diff={max_diff:.2e}')
print('All pass:', all_pass)


---
## Stage 3 — Training metric parity

Run a short training session (1 pretrain epoch + 5 finetune epochs) with identical seeds and compare validation metrics.

In [ ]:
# ── DGL: pretrain 1 epoch + finetune 5 epochs ─────────────
if PYG_ROOT in sys.path: sys.path.remove(PYG_ROOT)
if DGL_ROOT not in sys.path: sys.path.insert(0, DGL_ROOT)
for mod in list(sys.modules.keys()):
    if mod.startswith('txgnn'): del sys.modules[mod]
import txgnn as txgnn_dgl

torch.manual_seed(0); np.random.seed(0)
data_dgl2 = txgnn_dgl.TxData(data_folder_path=DATA_FOLDER)
data_dgl2.prepare_split(split=SPLIT, seed=SEED)

m_dgl = txgnn_dgl.TxGNN(data=data_dgl2, device=DEVICE)
m_dgl.model_initialize(n_hid=N_HID, n_inp=N_INP, n_out=N_OUT,
                        proto=PROTO, proto_num=PROTO_NUM, attention=ATTENTION,
                        sim_measure='all_nodes_profile', agg_measure='rarity')

torch.manual_seed(0)
m_dgl.pretrain(n_epoch=1, learning_rate=1e-3, batch_size=1024, train_print_per_n=9999)

torch.manual_seed(0)
m_dgl.finetune(n_epoch=5, learning_rate=1e-3, train_print_per_n=9999, valid_per_n=5)
print('DGL training done')


In [ ]:
# ── PyG: same config ───────────────────────────────────────
if DGL_ROOT in sys.path: sys.path.remove(DGL_ROOT)
if PYG_ROOT not in sys.path: sys.path.insert(0, PYG_ROOT)
for mod in list(sys.modules.keys()):
    if mod.startswith('txgnn'): del sys.modules[mod]
import txgnn as txgnn_pyg

torch.manual_seed(0); np.random.seed(0)
data_pyg2 = txgnn_pyg.TxData(data_folder_path=DATA_FOLDER)
data_pyg2.prepare_split(split=SPLIT, seed=SEED)

m_pyg = txgnn_pyg.TxGNN(data=data_pyg2, device=DEVICE)
m_pyg.model_initialize(n_hid=N_HID, n_inp=N_INP, n_out=N_OUT,
                        proto=PROTO, proto_num=PROTO_NUM, attention=ATTENTION,
                        sim_measure='all_nodes_profile', agg_measure='rarity')

torch.manual_seed(0)
m_pyg.pretrain(n_epoch=1, learning_rate=1e-3, batch_size=1024, train_print_per_n=9999)

torch.manual_seed(0)
m_pyg.finetune(n_epoch=5, learning_rate=1e-3, train_print_per_n=9999, valid_per_n=5)
print('PyG training done')


In [ ]:
# ── Compare validation metrics ─────────────────────────────
from sklearn.metrics import roc_auc_score, average_precision_score

# Switch to DGL for eval
if PYG_ROOT in sys.path: sys.path.remove(PYG_ROOT)
if DGL_ROOT not in sys.path: sys.path.insert(0, DGL_ROOT)
for mod in list(sys.modules.keys()):
    if mod.startswith('txgnn'): del sys.modules[mod]
import txgnn as txgnn_dgl
from txgnn.utils import evaluate_fb

dd_etypes = [
    ('drug', 'contraindication', 'disease'),
    ('drug', 'indication', 'disease'),
    ('drug', 'off-label use', 'disease'),
    ('disease', 'rev_contraindication', 'drug'),
    ('disease', 'rev_indication', 'drug'),
    ('disease', 'rev_off-label use', 'drug'),
]

(auroc_rel_d, auprc_rel_d, micro_auroc_d, micro_auprc_d, macro_auroc_d, macro_auprc_d), loss_d = \
    evaluate_fb(m_dgl.best_model, m_dgl.g_valid_pos, m_dgl.g_valid_neg,
                data_dgl2.G.to(DEVICE), dd_etypes, DEVICE)

print(f'DGL  Macro AUROC={macro_auroc_d:.4f}  Macro AUPRC={macro_auprc_d:.4f}  Loss={loss_d:.4f}')


In [ ]:
# Switch to PyG for eval
if DGL_ROOT in sys.path: sys.path.remove(DGL_ROOT)
if PYG_ROOT not in sys.path: sys.path.insert(0, PYG_ROOT)
for mod in list(sys.modules.keys()):
    if mod.startswith('txgnn'): del sys.modules[mod]
import txgnn as txgnn_pyg
from txgnn.utils import evaluate_fb as evaluate_fb_pyg

(auroc_rel_p, auprc_rel_p, micro_auroc_p, micro_auprc_p, macro_auroc_p, macro_auprc_p), loss_p = \
    evaluate_fb_pyg(m_pyg.best_model, m_pyg.g_valid_pos, m_pyg.g_valid_neg,
                    data_pyg2.G.to(DEVICE), dd_etypes, DEVICE)

print(f'PyG  Macro AUROC={macro_auroc_p:.4f}  Macro AUPRC={macro_auprc_p:.4f}  Loss={loss_p:.4f}')


In [ ]:
# ── Summary table ──────────────────────────────────────────
import pandas as pd

METRIC_TOL = 0.01   # allow ≤1 pp difference in AUROC/AUPRC

rows = []
for et in dd_etypes:
    if et in auroc_rel_d and et in auroc_rel_p:
        d_auc = auroc_rel_d[et]
        p_auc = auroc_rel_p[et]
        d_pr  = auprc_rel_d[et]
        p_pr  = auprc_rel_p[et]
        rows.append({
            'Edge type': str(et),
            'DGL AUROC': round(d_auc, 4),
            'PyG AUROC': round(p_auc, 4),
            'ΔAUROC': round(abs(d_auc - p_auc), 4),
            'DGL AUPRC': round(d_pr, 4),
            'PyG AUPRC': round(p_pr, 4),
            'ΔAUPRC': round(abs(d_pr - p_pr), 4),
            'Pass': abs(d_auc - p_auc) < METRIC_TOL and abs(d_pr - p_pr) < METRIC_TOL,
        })

rows.append({
    'Edge type': 'MACRO',
    'DGL AUROC': round(macro_auroc_d, 4),
    'PyG AUROC': round(macro_auroc_p, 4),
    'ΔAUROC': round(abs(macro_auroc_d - macro_auroc_p), 4),
    'DGL AUPRC': round(macro_auprc_d, 4),
    'PyG AUPRC': round(macro_auprc_p, 4),
    'ΔAUPRC': round(abs(macro_auprc_d - macro_auprc_p), 4),
    'Pass': abs(macro_auroc_d - macro_auroc_p) < METRIC_TOL,
})

df_summary = pd.DataFrame(rows)
print(df_summary.to_string(index=False))
print('\nAll pass:', df_summary['Pass'].all())


---
## Stage 4 — Disease-centric evaluation parity

Run the full disease-centric evaluation on a handful of test diseases and compare per-disease AUROC.

In [ ]:
# Take the first 5 diseases from the test set
SAMPLE_DISEASES = 5

# ── DGL eval ──────────────────────────────────────────────
if PYG_ROOT in sys.path: sys.path.remove(PYG_ROOT)
if DGL_ROOT not in sys.path: sys.path.insert(0, DGL_ROOT)
for mod in list(sys.modules.keys()):
    if mod.startswith('txgnn'): del sys.modules[mod]
import txgnn as txgnn_dgl

evaluator_dgl = txgnn_dgl.TxEval(model=m_dgl, data=data_dgl2)
disease_ids_dgl = evaluator_dgl.retrieve_disease_idxs_test_set('indication')[:SAMPLE_DISEASES]
result_dgl = evaluator_dgl.eval_disease_centric(
    disease_idxs=disease_ids_dgl.tolist(),
    relation='indication',
    return_raw=True,
    show_plot=False,
    verbose=False,
    simulate_random=False,
)
print('DGL disease-centric eval done for', len(disease_ids_dgl), 'diseases')


In [ ]:
# ── PyG eval ──────────────────────────────────────────────
if DGL_ROOT in sys.path: sys.path.remove(DGL_ROOT)
if PYG_ROOT not in sys.path: sys.path.insert(0, PYG_ROOT)
for mod in list(sys.modules.keys()):
    if mod.startswith('txgnn'): del sys.modules[mod]
import txgnn as txgnn_pyg

evaluator_pyg = txgnn_pyg.TxEval(model=m_pyg, data=data_pyg2)
disease_ids_pyg = evaluator_pyg.retrieve_disease_idxs_test_set('indication')[:SAMPLE_DISEASES]
result_pyg = evaluator_pyg.eval_disease_centric(
    disease_idxs=disease_ids_pyg.tolist(),
    relation='indication',
    return_raw=True,
    show_plot=False,
    verbose=False,
    simulate_random=False,
)
print('PyG disease-centric eval done for', len(disease_ids_pyg), 'diseases')


In [ ]:
# ── Compare per-disease AUROC ──────────────────────────────
auroc_d = result_dgl['result']['AUROC']
auroc_p = result_pyg['result']['AUROC']

print(f'{'Disease ID':>20s}  {'DGL AUROC':>10s}  {'PyG AUROC':>10s}  {'|Δ|':>8s}  Pass')
print('-' * 65)
TOL = 0.02
all_pass = True
for did in sorted(auroc_d.keys()):
    d = auroc_d[did]
    p = auroc_p.get(did, float('nan'))
    delta = abs(d - p) if not np.isnan(p) else float('nan')
    ok = delta < TOL if not np.isnan(delta) else False
    if not ok: all_pass = False
    print(f'{str(did):>20s}  {d:>10.4f}  {p:>10.4f}  {delta:>8.4f}  {"✓" if ok else "✗"}')

print('\nAll pass:', all_pass)


---
## Summary

If all stages pass:
- Stage 1 (graph construction) ✓ → same topology
- Stage 2 (forward pass) ✓ → same math, same weights
- Stage 3 (training metrics) ✓ → converges to same validation performance
- Stage 4 (disease-centric) ✓ → same ranking quality per disease

The PyG refactoring is logically equivalent to the DGL implementation.